# Minecraft Fine-tuning (Qwen2.5-7B-Instruct / RTX 5060 Ti)

## 사전 준비

```bash
python -m venv venv
source venv/Scripts/activate

# RTX 5060 Ti (Blackwell) — CUDA 12.8 필수
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

pip install jupyter
python -m ipykernel install --user --name venv --display-name "mine-tuning (venv)"
```

## 0. 환경 확인

In [ ]:
import subprocess
import sys
import torch

try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    sys.exit('NVIDIA 드라이버를 찾을 수 없습니다.')

if not torch.cuda.is_available():
    sys.exit('CUDA를 사용할 수 없습니다. cu128 버전 PyTorch가 설치되어 있는지 확인하세요.')

props = torch.cuda.get_device_properties(0)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {props.total_memory / 1024**3:.1f} GB')
print(f'bf16    : {torch.cuda.is_bf16_supported()}')

## 1. 라이브러리 설치

In [ ]:
import subprocess

subprocess.run([
    'pip', 'install', '-q',
    'transformers==4.51.3',
    'trl==0.17.0',
    'datasets',
    'peft',
    'accelerate',
    'bitsandbytes',
], check=True)

print('패키지 설치 완료')

In [ ]:
!pip freeze > requirements-ft.txt

## 2. 데이터 준비

In [ ]:
from datasets import load_dataset

# 모델에게 어떻게 답변할지 방향을 잡아주는 시스템 프롬프트
# 단순히 'Minecraft expert'라고만 하는 것보다 구체적인 지침을 주는 게 특화 성능에 훨씬 효과적
SYSTEM_PROMPT = """You are an expert Minecraft assistant with deep knowledge of all game mechanics, crafting recipes, biomes, mobs, and strategies.

When answering:
- Be specific and accurate about game mechanics
- Include exact crafting recipes when relevant (materials and placement)
- For survival tips, prioritize safety and efficiency
- Keep answers clear and beginner-friendly unless the question is advanced"""

def format_instruction(example):
    # Qwen 계열 모델의 ChatML 포맷: <|im_start|> role \n content <|im_end|>
    return {
        'text': (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{example['question']}<|im_end|>\n"
            f"<|im_start|>assistant\n{example['answer']}<|im_end|>"
        )
    }

ds = load_dataset('ddorin/minecraft-question-answer-260k')
train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])

print(f'데이터 수: {len(train_data):,}개')
print('\n--- 샘플 ---')
print(train_data[0]['text'])

## 3. 모델 로드 (4bit 양자화)

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# 4bit 양자화: 7B 모델을 ~5.5GB로 압축
# nf4 + double_quant 조합이 품질 손실 최소화에 가장 효과적
# compute_dtype은 bf16 — RTX 5060 Ti (Blackwell)에서 fp16보다 안정적
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},  # 전체 모델을 GPU 0번에 로드
)

# gradient_checkpointing: 활성화값을 재계산해서 VRAM 절약 (속도는 약간 감소)
# use_reentrant=False: PyTorch 2.9+ 권장 방식, 경고 메시지 억제
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
model.enable_input_require_grads()

used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'모델 로드 완료 | VRAM {used:.1f} / {total:.1f} GB')

## 4. LoRA 설정

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# LoRA: 전체 모델 대신 작은 어댑터(~0.5%)만 학습 → VRAM 절약 + 빠른 학습
# r=16: rank. 높을수록 표현력↑ VRAM↑. 도메인 특화에는 16이 적절
# lora_alpha=32: 학습률 스케일. 통상 r의 2배로 설정
# target_modules: attention + FFN 레이어 전부 커버 (attention만 하는 것보다 효과적)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 전체의 약 0.5~1% 학습

## 5. 학습

In [ ]:
import os
from datetime import datetime
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig

date_str   = datetime.now().strftime('%Y%m%d_%H%M')
OUTPUT_DIR = rf'C:\mine-tuning-output\Qwen2.5-7B_{date_str}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                 # 1에포크는 loss가 수렴 전에 끝남 → 3으로
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # 실질 배치 크기 = 4 * 4 = 16
    learning_rate=2e-4,
    bf16=True,                          # RTX 5060 Ti (Blackwell) 권장
    fp16=False,                         # bf16 사용 시 반드시 False
    optim='adamw_bnb_8bit',             # 8bit 옵티마이저 → VRAM 절약
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,                  # 전체 스텝의 3%를 워밍업에 사용
    packing=True,                       # 짧은 샘플들을 하나의 시퀀스로 묶어 처리 → 속도 30~40%↑
    max_seq_length=512,
    logging_steps=50,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=3,                 # 체크포인트 최대 3개 보관 (디스크 절약)
    dataloader_num_workers=0,           # Windows 멀티프로세싱 이슈 방지
    report_to='none',
)

# 에포크 완료 시마다 별도 폴더에 저장 (나중에 에포크별 성능 비교 가능)
class SaveEpochCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch     = int(state.epoch)
        save_path = os.path.join(OUTPUT_DIR, f'epoch-{epoch}')
        trainer.model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f'에포크 {epoch} 저장 완료 → {save_path}')

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    processing_class=tokenizer,
    callbacks=[SaveEpochCallback()],
)

print(f'학습 데이터: {len(train_data):,}건 | 에포크: {training_args.num_train_epochs}')
trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'학습 완료 → {OUTPUT_DIR}')

## 6. 버전 확인

In [ ]:
import transformers, trl, torch

print(f'transformers : {transformers.__version__}')
print(f'trl          : {trl.__version__}')
print(f'torch        : {torch.__version__}')
print(f'CUDA         : {torch.version.cuda}')